# EDA 004 · Results-day move training set

**The question.** How big will the price jump be when this company reports its results?

**One row** is a company on the evening before a results release (an 8-K with item 2.02,
2010 – 2022), with the features known at that close. **The answer**, `move`, is the absolute
return from that close to the close after the 8-K was filed, minus the market's (SPY) return over
the same days: 0.08 means the stock moved about 8% more than the market, up or down. **The naive
baseline**, `prev4_mean`, is the mean of the company's previous four such moves (`n_prev` says how
many it had). The model (006) has to beat it.

Timing columns: `report_date` (the day results were released), `filing_date` (the day the 8-K was
filed, sometimes a few days later), `close_after` (the close after filing; the window's end).

**Words used below**

- **Row**: one company on one date, with everything known about it at that day's close (the
  *features*) and what happened next (the *answer*).
- **Eligible**: the rows a model may learn from: the company filed something with the SEC in the
  last 12 months, is not a fund or blank-check company, its price is at least \$1, it trades at
  least \$10,000 a day, and it has at least 63 days of prices.
- **Development rows**: rows up to 2022 with an answer. Rows from 2023 on are the sealed holdout
  and are never looked at here.
- **Test years**: 2019, 2020, 2021, 2022. Each is predicted by a model trained only on earlier
  years ("walk-forward").
- **Purge**: a training row whose answer window reaches into the test year is dropped, so the
  model never trains on days it is tested on.
- **Spearman correlation (ρ)**: how well two columns rank alike, from −1 to +1; 0 means no order in
  common. It ignores scale, so it is safe with long tails.
- **Decile**: one of ten equal-sized groups after sorting by a feature (decile 1 = the lowest 10%).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sentinel import eda

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:,.4g}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
TEST_YEARS = eda.TEST_YEARS

## 1. Load

The development rows only (`as_of` on or before 2022-12-31).

In [ ]:
SET, ANSWER = "results_move", "move"
df = eda.load(SET)
FEATURES = eda.features(df)
print(f"{len(df):,} rows, {df['cik'].nunique():,} companies, {df.shape[1]} columns, "
      f"{len(FEATURES)} features")
print("dates:", df["as_of"].min().date(), "to", df["as_of"].max().date())
print("duplicate (cik, as_of) keys:", int(df.duplicated(["cik", "as_of"]).sum()))
ok = df[df["eligible"] & df[ANSWER].notna()].copy()  # what the models learn from
print(f"eligible with an answer: {len(ok):,} rows ({len(ok) / len(df):.1%}), "
      f"{ok['cik'].nunique():,} companies")

## 2. Columns by family

In [ ]:
fam = pd.Series({c: eda.family(c) for c in FEATURES}, name="family")
display(fam.value_counts().rename("features").to_frame())
display(fam.groupby(fam).apply(lambda s: ", ".join(s.index)).rename("columns").to_frame())
others = [c for c in df.columns if c not in FEATURES]
print("not features:", ", ".join(others))
display(df.dtypes.astype(str).value_counts().rename("columns").to_frame())

## 3. Coverage

Who is in the data, per year and per kind of company. *Coverage level*: `full` (files with the
SEC and has prices), `foreign` (a foreign filer: 20-F or 40-F reports), `price_only` (trades in
the US, files nothing periodic). Size bands by market value: micro below \$300 M, small to
\$2 B, mid to \$10 B, large above; `unknown` where market value is missing.

In [ ]:
year = df["as_of"].dt.year
by_year = eda.coverage_table(df, year, "move")
display(by_year)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
pd.DataFrame({"eligible": df[df["eligible"]].groupby(year[df["eligible"]]).size(),
              "not eligible": df[~df["eligible"]].groupby(year[~df["eligible"]]).size()}
             ).plot.bar(stacked=True, ax=ax[0], title="rows per year")
by_year["companies"].plot.bar(ax=ax[1], title="companies per year")
plt.tight_layout(); plt.show()

In [ ]:
for col in ["coverage", "filer_type", "foreign_filer", "sector"]:
    display(eda.coverage_table(df, col, "move").sort_values("rows", ascending=False))
df["size_band"] = eda.size_band(df["market_cap"])
ok["size_band"] = eda.size_band(ok["market_cap"])
display(eda.coverage_table(df, "size_band", "move"))

Why rows are **not eligible** (a row can fail several rules):

In [ ]:
display(eda.eligibility_reasons(df))

Releases per company per year (about four is normal: three quarters and the annual report):

In [ ]:
per = ok.groupby([ok["cik"], ok["as_of"].dt.year]).size()
display(per.value_counts().sort_index().rename("company-years").to_frame())

## 4. Missing values

Share of eligible rows with an answer where each feature is missing. A missing value is not
an error when the fact does not exist for that company (constitution II: missing, never zero):
foreign filers do not file Form 4 or 8-K; companies without financial filings have no ratios; a
ratio with a zero denominator is left empty.

In [ ]:
miss = eda.missing_table(ok, FEATURES)
print(f"features with no missing values: {(miss['missing'] == 0).sum()}; "
      f"over 50% missing: {(miss['missing'] > 0.5).sum()}")
display(miss[miss["missing"] > 0])
display(miss.groupby("family")["missing"].agg(["mean", "max"]).sort_values("mean"))

The most-missing features by filer type, coverage level and year: where the gaps come from.

In [ ]:
top = miss.index[:15].tolist()
display(ok.groupby("filer_type")[top].apply(lambda g: g.isna().mean()).T)
display(ok.groupby("coverage")[top].apply(lambda g: g.isna().mean()).T)
display(eda.missing_by_year(ok, top))

## 5. Impossible and extreme values

`infinite`: values of ±∞ (a division by zero that slipped through); `negative`: values below
zero where that is impossible (counts, prices, volatilities, amounts). Then each feature's spread:
the 1st percentile (p1), median and 99th percentile (p99) against its minimum and maximum: a
maximum far beyond p99 means a long tail or a data error.

In [ ]:
ext = eda.extremes_table(ok, FEATURES)
bad = ext[(ext["infinite"] > 0) | (ext["negative"] > 0)]
print("features with infinite or impossible negative values:", len(bad))
display(bad)
spread = ext.assign(max_over_p99=ext["max"] / ext["p99"].where(ext["p99"] > 0))
display(spread.sort_values("max_over_p99", ascending=False).head(25))

**Suspect values.** Specific checks behind the long tails above. A foreign filer's market value
should be missing (its share count is of ordinary shares while its US price is often an ADR's,
risk_params v5); no US company was worth \$5 trillion before 2023 (Apple's peak was about \$3
trillion); a 12-month insider total above \$100 billion is a typo in a filing.

In [ ]:
checks = {
    "foreign filer with a market value": ok["foreign_filer"] & ok["market_cap"].notna(),
    "market value 0": ok["market_cap"] == 0,
    "market value above 5 trillion dollars": ok["market_cap"] > 5e12,
    "shares outstanding above 50 billion": ok["shares_outstanding"] > 5e10,
    "one-month return above +1,000%": ok["ret_1m"] > 10,
    "12-month return above +2,000%": ok["ret_12_1"] > 20,
    "insider sales above 100 billion dollars in 12 months": ok["sell_value_12m"] > 1e11,
    "insider buys above 10 billion dollars in 12 months": ok["buy_value_12m"] > 1e10,
    "cash above total assets": ok["cash_to_assets"] > 1,
    "negative leverage": ok["leverage"] < 0,
    "negative book value (book_to_market < 0)": ok["book_to_market"] < 0,
    "results overdue by more than a year": ok["days_to_next_results"] < -365,
}
display(pd.DataFrame({k: {"rows": int(v.sum()), "companies": ok.loc[v, "cik"].nunique(),
                          "share": v.mean()} for k, v in checks.items()}).T)

In [ ]:
print("answers < 0:", int((ok[ANSWER] < 0).sum()), "| exactly 0:", int((ok[ANSWER] == 0).sum()))
for t in [0.2, 0.3, 0.5, 1.0]:
    print(f"moves above {t:.0%}: {(ok[ANSWER] > t).sum():,} ({(ok[ANSWER] > t).mean():.2%})")

## 6. The answer

The move is a size (always ≥ 0) with a long tail: most results days are quiet, a few are
dramatic. Shown on a log scale, by year, sector and size.

In [ ]:
display(ok[ANSWER].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).to_frame())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
np.log10(ok[ANSWER].clip(lower=1e-4)).plot.hist(bins=100, ax=ax[0], title="log10 of move")
q = ok.groupby(ok["as_of"].dt.year)[ANSWER].quantile([0.25, 0.5, 0.75, 0.9]).unstack()
q.plot(ax=ax[1], title="25th, 50th, 75th, 90th percentile by year", marker="o")
plt.tight_layout(); plt.show()
for col in ["sector", "size_band", "coverage", "filer_type"]:
    display(ok.groupby(col)[ANSWER].agg(["count", "median", "mean"]).sort_values("median"))

**Timing.** Days from the release (`report_date`) to the 8-K's filing and to the window's end.
A long gap means the window includes more than the results reaction.

In [ ]:
t = pd.DataFrame({
    "release to filing": (ok["filing_date"] - ok["report_date"]).dt.days,
    "eve to release": (ok["report_date"] - ok["as_of"]).dt.days,
    "eve to window end": (ok["close_after"] - ok["as_of"]).dt.days,
})
display(t.describe(percentiles=[0.5, 0.9, 0.99]).T)
display((t["release to filing"].clip(upper=5).value_counts().sort_index()
         .rename("releases (5 = 5 or more days)")).to_frame())
display(ok.groupby(t["eve to window end"].clip(upper=8))[ANSWER].median()
        .rename("median move by window length in days (8 = 8+)").to_frame())

## 7. The naive baseline

`prev4_mean` exists when the company had earlier moves (`n_prev` ≥ 1). Compared with a floor
that knows nothing about the company: the median move of all earlier years (a constant).
Errors are absolute (0.01 = one percentage point). Spearman: does the baseline rank the
releases right?

In [ ]:
display(ok["n_prev"].value_counts().sort_index().rename("releases by n_prev").to_frame())
b = ok[ok["prev4_mean"].notna()]
years = b["as_of"].dt.year

def skill(f, floor):
    e, c = f[ANSWER] - f["prev4_mean"], f[ANSWER] - floor
    return pd.Series({"rows": len(f), "MAE baseline": e.abs().mean(), "MedAE baseline": e.abs().median(),
                      "MAE constant": c.abs().mean(), "bias baseline": e.mean(),
                      "spearman": f["prev4_mean"].rank().corr(f[ANSWER].rank())})

rows = {}
for y in sorted(years.unique()):
    floor = ok.loc[ok["close_after"] < f"{y}-01-01", ANSWER].median()
    rows[y] = skill(b[years == y], floor if pd.notna(floor) else b[ANSWER].median())
display(pd.DataFrame(rows).T)
display(b.groupby(b["n_prev"].clip(upper=4))
        .apply(lambda f: f["prev4_mean"].rank().corr(f[ANSWER].rank()))
        .rename("spearman by n_prev (4 = 4)").to_frame())

## 8. Feature–answer relations

Spearman ρ of every feature with the move, on eligible rows with an answer; strongest
first. `n` is the rows where the feature is present.

In [ ]:
ranks = eda.rank_table(ok, FEATURES, "move")
display(ranks)
display(ranks.groupby("family")["spearman"].apply(lambda s: s.abs().max()).sort_values()
        .rename("strongest |ρ| in family").to_frame())

The answer by decile of the ten strongest features (mean of the answer in each tenth):

In [ ]:
top10 = ranks.index[:10].tolist()
fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharey=True)
for ax, col in zip(axes.flat, top10):
    d = eda.decile_table(ok, col, "move")
    ax.plot(d.index.astype(int), d["mean"], marker="o")
    ax.set_title(col, fontsize=9); ax.set_xticks(range(1, 11))
plt.suptitle("mean answer by feature decile (1 = lowest)"); plt.tight_layout(); plt.show()

**Stability over time.** The same ρ within each year for the 15 strongest features. A relation
that flips sign or fades in some years is less trustworthy than its pooled value suggests.

In [ ]:
top15 = ranks.index[:15].tolist()
yearly = eda.rank_by_year(ok, top15, "move")
display(yearly)
pooled = ranks.loc[top15, "spearman"]
flips = (np.sign(yearly).ne(np.sign(pooled), axis=0) & yearly.notna()).sum(axis=1)
display(pd.DataFrame({"pooled": pooled, "min": yearly.min(axis=1), "max": yearly.max(axis=1),
                      "years with the other sign": flips}))

**What adds to the baseline.** Which features relate to the move *after* the company's own
history: the ratio log(move / prev4_mean), on rows with a baseline and a non-zero move.

In [ ]:
s = b[(b[ANSWER] > 0) & (b["prev4_mean"] > 0)]
s = s.assign(surprise=np.log(s[ANSWER] / s["prev4_mean"]))
display(eda.rank_table(s, FEATURES, "surprise").head(25))

## 9. Redundant features

Pairs of features that rank rows almost identically (|ρ| ≥ 0.9, on a fixed sample of 200,000
rows). Tree models cope with them, but they split importance between the pair and make the model
harder to explain; linear models need one of each pair dropped.

In [ ]:
pairs = eda.redundant_pairs(ok, FEATURES)
print(len(pairs), "pairs")
display(pairs)

## 10. Leakage and purging

*Leakage* is information from after the row's date reaching its features: a model then looks
brilliant in testing and fails live. Checks: (a) every answer window ends after the row date;
(b) no feature relates to the answer implausibly well (|ρ| ≥ 0.95); (c) how many training rows
the purge removes before each test year, and how many remain.

In [ ]:
gap = (ok["close_after"] - ok["as_of"]).dt.days
print("windows ending on or before the row date:", int((gap <= 0).sum()))
display(gap.describe(percentiles=[0.01, 0.5, 0.99]).rename("days from row to window end").to_frame())
print("features with |rho| >= 0.95:", ranks.index[ranks["spearman"].abs() >= 0.95].tolist())
display(eda.purge_table(ok, "close_after"))

Also: no feature row is dated on or after its release, and `prev4_mean` uses only earlier windows:

In [ ]:
print("rows with as_of on or after report_date:", int((ok["as_of"] >= ok["report_date"]).sum()))
prev_end = ok.groupby("cik")["close_after"].shift(1)
print("rows whose previous release's window ends after this row's date:",
      int((prev_end > ok["as_of"]).sum()))

## 11. Findings

*Export of 30 October 2026. Numbers are from the sections named in brackets; all are aggregates.*

### In one paragraph

The results-move set is clean and leak-free: 157,566 releases (5,947 companies) are eligible with
an answer, 92% of all rows. The surprise is the baseline: the mean of a company's last four moves
*ranks* releases (rank correlation 0.30–0.39) but its *size* is no more accurate than one constant
for everyone. Past volatility ranks the moves just as well as the company's own history. The model
should be judged on ranking and on bands, not on average error alone.

### What is clean

- **Coverage** [3]: 2010–2022, 10,000–16,000 releases a year, 92% eligible; most companies have
  four releases a year (33,310 company-years with four).
- **No foreign filers**: they do not file 8-K item 2.02, so the results move covers US filers
  (plus 114 rows of price-only stocks). This is by definition, not a gap.
- **No leakage** [10]: every row is dated before its release; each baseline uses only earlier
  windows; windows end 2–14 days after the row. Because windows are short, the purge removes at
  most 5 rows per test year.
- **Stable relations** [8]: none of the 15 strongest features changes direction in any year.

### The answer [6]

- Median move 3.7% beyond the market, mean 5.8%; 90% of moves are below 13.5%, 99% below 31%.
  276 moves exceed 50%, 26 exceed 100%; 5 are exactly 0.
- Utilities move least (median 1.8%), technology most (5.4%); large companies 2.9%, small and
  micro 4.1–4.4%.
- **Timing**: 84% of the 8-Ks are filed on the release day, 1.9% (2,948) five or more days later.
  The longer the window, the larger the median move (3.8% for 2 days, 5.2% for 8 or more): long
  windows include other news.

### The naive baseline [7]

| | 2019 | 2020 | 2021 | 2022 |
|---|---|---|---|---|
| mean absolute error, baseline | 0.0455 | 0.0513 | 0.0470 | 0.0516 |
| mean absolute error, one constant (earlier median) | 0.0454 | 0.0481 | 0.0416 | 0.0517 |
| rank correlation, baseline | 0.39 | 0.30 | 0.31 | 0.34 |

The baseline knows *which* companies move more, but a mean of four skewed moves is noisy, and
the mean absolute error favours a median. Its ranking improves with history: rank correlation 0.21
with one earlier move, 0.37 with four. 1,355 releases (each company's first) have no baseline.

### What relates to the answer [8]

- **Volatility features rank the move as well as the company's history**: `idio_vol_252`,
  `range_vol_21`, `vol_252`, `vol_63` (ρ 0.33 each), the fall from the 12-month high (−0.32),
  distance from the high (−0.27).
- Then beta (0.18), loss-making (0.18), cash share of assets (0.18), price level (−0.15), size
  (−0.12), leverage (−0.12).
- **Beyond the company's history** (log of move over baseline), every feature is weak (|ρ| ≤
  0.065): recent returns (−0.06), credit spread change (+0.06), VIX (+0.04). The gain will come
  from combining history with volatility, not from a hidden signal.

### Missing values [4]

Financial features are mostly missing in 2010–2011 (market value missing for 78% of 2010 rows,
48% in 2011, 6–8% from 2012), because small companies only started tagging their reports
(XBRL) in 2011. The other two sets start in 2012 and do not have this.

### Redundant features [9]

13 pairs with |ρ| ≥ 0.9, the same as in the volatility set, plus traded value with market value.

### Proposed cleaning rules (for docs/decisions.md; fixed in the pipeline with tests)

The same share-count and insider problems as in the volatility set (one rule fixes all three
sets):

1. **Impossible share counts**: 65 rows (39 companies) with a market value above \$5 trillion,
   125 rows with more than 50 billion shares, 228 rows (105 companies) with a market value of 0,
   and one row with a **negative** share count. Proposal as in the volatility notebook: such a
   count is missing. [5]
2. **Insider dollar totals**: 36 rows above \$100 billion of sales, 39 above \$10 billion of buys.
   [5]

### Advice for the results-move model (006)

- Compare against two baselines: the pre-registered mean of the previous four, and a stronger
  "rule" level such as the *median* of the previous four, shrunk toward the sector median when a
  company has few releases.
- Judge by rank correlation and by the bands (D62: chances of a move above 5%, 10%, 20%), not by
  mean absolute error alone, which a constant already matches.
- Model the log of the move (or a quantile loss): the tail is long.
- Key features: past volatility (one of each redundant pair), drawdown, distance from the high,
  beta, loss-making, cash, size, and the baseline itself (declare `prev4_mean` and `n_prev` as
  features, as docs/plan.md allows).
- Consider training from 2012, or accept that financial features are missing in 2010–2011.
- Check the test years with and without windows of 5 or more days; if long windows hurt, record
  it (the window rule is pre-registered and stays).

### Open questions for the researcher

- 69 company-years have 5 or 6 releases. Are some of them a release and its supplement counted
  twice? Worth checking a few named cases.